In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_path = ('/content/drive/MyDrive/FAKE-1-XYZ-0001-SES.las')

import os
if os.getcwd() != drive_path :
  os.chdir(drive_path)

In [ ]:
!pip install lasio

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import lasio
import numpy as np

In [ ]:
def prepare_las(las):
    lasdf = las.df()

    # Use this cell if you need to convert the index depth column into a new regular column
    # It will depend on the LAS file structure
    lasdf = lasdf.rename_axis('MD').reset_index()

    # Select columns that start with "T2_DIST"
    columns_to_sum = [col for col in lasdf.columns if col.startswith(('T2_DIST','T2DIST'))]

    # Row-wise sum: Summing across rows for the selected columns
    lasdf['NMRTT'] = lasdf[columns_to_sum].sum(axis=1)

    # Column-wise sum: Summing down each selected column
    column_sum = lasdf[columns_to_sum].sum()

    # Drop the original "T2_DIST" columns
    lasdf.drop(columns=columns_to_sum, inplace=True)

    return lasdf

def convert_to_meters_dataframe(df, columns_to_convert, conversion_factor=0.3048):
    for col in columns_to_convert:
        if col in df.columns:
            df[col] = df[col] * conversion_factor
    return df

def plot_graficos(lasdf, las):

    # Check and convert the 'MD' depth column to meters
    if 'MD' in lasdf.columns:
        if lasdf['MD'].max() > 10000:  # Assumes values in feet are much larger
            lasdf = convert_to_meters_dataframe(lasdf, ['MD'])

    # Fig histograms
    fig_hist = plt.figure(figsize=(7, 7))

    # Select columns that start with 'DT', 'DTC', or 'DTCO'
    columns_to_read = [col for col in lasdf.columns if col.startswith(('DT', 'DTC', 'DTCO'))]

    # Calculate Vp
    def calculo_vp(dt, unidade_dt):
       if not unidade_dt or unidade_dt.lower() in ['', 'none']:
        unidade_dt = 'us/ft'

       unidade_dt = unidade_dt.lower()

       if unidade_dt.lower() in ['us/ft', 'µs/ft']:
        return 304800 / dt  # converte para m/s
       elif unidade_dt.lower() in ['us/m', 'µs/m']:
        return 1e6 / dt
       elif unidade_dt.lower() in ['s/m']:
        return 1 / dt
       else:
        print("DT unit not found")
        return np.nan

    if columns_to_read:
       dt_column = columns_to_read[0]
       dt_unidade = las.curves[dt_column].unit
       lasdf_filtered = lasdf[['MD', dt_column]].dropna()
       vp_values = calculo_vp(lasdf_filtered[dt_column], dt_unidade)
       lasdf.loc[lasdf_filtered.index, 'VP'] = vp_values
       lasdf = lasdf.dropna(subset=['VP'])
    else:
       print("'DT', 'DTC' or 'DTCO' column not found in the DataFrame.")

    if columns_to_read:
        dt_column = columns_to_read[0] # Use the first column found

    # Specify the columns you want to plot
    columns_to_plot = columns_to_read + ['NPHI', 'RHOB', 'VP', 'GR']

    # Filter columns_to_plot to only include columns that are in the DataFrame
    available_columns = [col for col in columns_to_plot if col in lasdf.columns]

    # Plot histograms for specified columns only
    if available_columns:
        lasdf[available_columns].hist(ax=fig_hist.gca())
    else:
        print("'DT', 'DTC', 'DTCO', 'NPHI', 'RHOB', 'VP' or 'GR' column not found in the DataFrame")

    # Fig Depth x Logs
    fig = plt.figure(figsize=(10, 10))

    if columns_to_read:
        dt_column = columns_to_read[0]

        # Remove NaNs
        unidade_dt = las.curves[dt_column].unit
        lasdf_filtered = lasdf[['MD', dt_column]].dropna()
        depth_values = lasdf_filtered['MD'].values
        vp_values = calculo_vp(lasdf_filtered[dt_column], unidade_dt)

        # Plot Depth x Vp
        ax1 = fig.add_subplot(141)
        ax1.plot(vp_values, depth_values, '-b')
        ax1.set_xlabel('m/s')
        ax1.set_ylabel('Depth (m)', labelpad=20)
        ax1.set_title('VP')
        ax1.invert_yaxis()
    else:
        print("'DT', 'DTC' ou 'DTCO' column not found in the DataFrame")

    # Plot RHOB
    if 'RHOB' in lasdf.columns:
        ax2 = fig.add_subplot(142)
        ax2.plot(lasdf['RHOB'], lasdf['MD'], 'g-')
        ax2.set_xlabel('g/cm³')
        ax2.set_title('RHOB')
        ax2.invert_yaxis()
    else:
        print("'RHOB' column not found in the DataFrame")

    # Plot NPHI
    if 'NPHI' in lasdf.columns:
        ax3 = fig.add_subplot(143)
        ax3.plot(lasdf['NPHI'], lasdf['MD'], 'r-')
        ax3.set_xlabel('v/v')
        ax3.set_title('NPHI')
        ax3.invert_yaxis()
    else:
        print("'NPHI' column not found in the DataFrame")

    # Plot GR
    if 'GR' in lasdf.columns:
        ax4 = fig.add_subplot(144)
        ax4.plot(lasdf['GR'], lasdf['MD'], 'k-')
        ax4.set_xlabel('API')
        ax4.set_title('GR')
        ax4.invert_yaxis()
    else:
        print("'GR' column not found in the DataFrame")

    plt.tight_layout()
    plt.show()

In [ ]:
las = lasio.read('FAKE-1-XYZ-0001-SES.las')
lasdf_XYZ0001 = prepare_las(las)
lasdf_XYZ0001.describe()

In [ ]:
lasdf = lasdf_XYZ0001
pltdf_XYZ0001 = plot_graficos(lasdf,las)
plt.show(pltdf_XYZ0001)